In [3]:
!git clone https://github.com/lorenzo-stacchio/Deep-Armocromia.git

Cloning into 'Deep-Armocromia'...


In [10]:
import io
import time
import base64
import re
import logging
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from tqdm import tqdm
from PIL import Image
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# ====================================================
# 로깅 설정
# ====================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("benchmark_eval.log", encoding="utf-8")
    ]
)
logger = logging.getLogger(__name__)

# ====================================================
# 1. 환경 설정
# ====================================================
# ⚠️ [사용자 재설정 필요] RunPod Pod ID가 바뀌면 아래 URL 수정 필요
RUNPOD_BASE_URL = "https://agm1l8bdfahw42-8000.proxy.runpod.net/v1"
API_KEY = "qwenbench123"
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

# ⚠️ [사용자 재설정 필요] 로컬 데이터셋 경로가 다르면 수정 필요
BASE_DIR = Path("./Deep-Armocromia/dataset")
ARMOCROMIA_CSV = BASE_DIR / "annotations.csv"

# ====================================================
# 태스크별 max_tokens 설정
# ====================================================
TASK_MAX_TOKENS = {
    "Color Recognition": 20,
    "Color Extraction":  50,
    "Color Proportion":  80,
    "Color Comparison":  80,
    "Robustness":        20,
    "Armocromia":        10,
}
DEFAULT_MAX_TOKENS = 80

client = OpenAI(base_url=RUNPOD_BASE_URL, api_key=API_KEY)

# ====================================================
# 2. 유틸리티 함수
# ====================================================
def encode_image(img_input) -> str:
    """이미지 경로 또는 PIL 객체를 Base64로 인코딩 (토큰 초과 방지 리사이즈 포함)"""
    if isinstance(img_input, (str, Path)):
        img = Image.open(img_input)
    else:
        img = img_input
    if img.mode != "RGB":
        img = img.convert("RGB")
    # 512px 초과 시 리사이즈 → 토큰 초과(400 Bad Request) 방지
    max_size = 512
    if img.width > max_size or img.height > max_size:
        img.thumbnail((max_size, max_size), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


@retry(
    retry=retry_if_exception_type(Exception),
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    reraise=False
)
def _call_api(img_b64: str, question: str, max_tokens: int) -> str:
    """실제 API 호출 (재시도 로직 포함)"""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                ]
            }
        ],
        max_tokens=max_tokens,
        temperature=0,
        timeout=180
    )
    return response.choices[0].message.content.strip()


def ask_vlm(question: str, img_input, task_name: str = "default") -> tuple[str, float]:
    """
    VLM API 호출 및 응답 반환
    - 태스크별 max_tokens 자동 적용
    - 재시도 3회
    - 실패 시 None 반환 (분모 제외 처리를 위해)
    """
    max_tokens = TASK_MAX_TOKENS.get(task_name, DEFAULT_MAX_TOKENS)
    try:
        img_b64 = encode_image(img_input)
        start = time.time()
        result = _call_api(img_b64, question, max_tokens)
        elapsed = time.time() - start
        return result, elapsed
    except Exception as e:
        logger.warning(f"[API 최종 실패] task={task_name}, error={e}")
        return None, 0.0


# ====================================================
# 3. 정확도 평가 로직
# ====================================================
def normalize_answer(text: str) -> str:
    """답변 텍스트 정규화"""
    text = text.lower().strip()
    text = text.split("\n")[0].strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_correct(pred: str, answer: str, task_name: str = "") -> bool:
    """
    정답 여부 판별
    - 정확 매칭 우선
    - 부정 표현 감지 ("not red", "isn't spring" 등)
    - 단어 경계 매칭
    """
    if not pred or not answer:
        return False

    pred_norm = normalize_answer(pred)
    ans_norm = normalize_answer(answer)

    # 1) 정확 매칭
    if pred_norm == ans_norm:
        return True

    # 2) 부정 표현 감지
    negation_patterns = [
        rf"\bnot\s+{re.escape(ans_norm)}\b",
        rf"\bisnt\s+{re.escape(ans_norm)}\b",
        rf"\bno\s+{re.escape(ans_norm)}\b",
        rf"\bnever\s+{re.escape(ans_norm)}\b",
    ]
    for pat in negation_patterns:
        if re.search(pat, pred_norm):
            return False

    # 3) 단어 경계 매칭
    if re.search(rf"\b{re.escape(ans_norm)}\b", pred_norm):
        return True

    return False


# ====================================================
# 4. ColorBench 종합 평가
# ====================================================
def eval_colorbench_full() -> dict:
    """ColorBench 평가 — 전체 데이터셋"""
    logger.info("🚀 [1/2] ColorBench 종합 평가 시작 (전체 데이터셋)...")

    ds = load_dataset("umd-zhou-lab/ColorBench", split="test")

    target_tasks = [
        "Color Recognition",
        "Color Extraction",
        "Color Proportion",
        "Color Comparison",
        "Robustness",
    ]

    task_results = {}

    for task_name in target_tasks:
        task_ds = [row for row in ds if row.get("task") == task_name]
        if not task_ds:
            logger.warning(f"⚠️ Task [{task_name}] 데이터 없음, 스킵")
            continue

        samples = task_ds
        logger.info(f"📊 Task: [{task_name}] — 전체 {len(samples)}개 평가 중...")

        correct = 0
        valid = 0
        failed_api = 0

        for row in tqdm(samples, desc=task_name, leave=False):
            question = row.get("question", "")
            answer = str(row.get("answer", "")).strip().lower()
            if not row.get("image") or not answer:
                continue

            pred, elapsed = ask_vlm(question, row["image"], task_name=task_name)

            if pred is None:
                failed_api += 1
                continue

            valid += 1
            if is_correct(pred, answer, task_name):
                correct += 1

            logger.debug(f"  Q: {question[:60]} | GT: {answer} | Pred: {pred[:40]} | {'✅' if is_correct(pred, answer) else '❌'}")

        acc = (correct / valid * 100) if valid > 0 else 0.0
        task_results[task_name] = {
            "accuracy": acc,
            "correct": correct,
            "valid": valid,
            "failed_api": failed_api,
            "total": len(samples),
        }
        logger.info(f"  ✅ [{task_name}] 정확도: {acc:.2f}% ({correct}/{valid}, API 실패: {failed_api})")

    return task_results


# ====================================================
# 5. Deep-Armocromia 평가
# ====================================================
def eval_armocromia() -> dict:
    """Armocromia 퍼스널 컬러 평가 — 전체 test 파티션"""
    logger.info("\n🚀 [2/2] Deep-Armocromia 실전 진단 평가 시작 (전체 데이터셋)...")

    if not ARMOCROMIA_CSV.exists():
        logger.error(f"⚠️ CSV를 찾을 수 없습니다: {ARMOCROMIA_CSV}")
        return {}

    df = pd.read_csv(ARMOCROMIA_CSV)
    test_df = df[df["partition"] == "test"].copy()

    if test_df.empty:
        logger.error("⚠️ 'test' 파티션 데이터가 없습니다.")
        return {}

    test_samples = test_df
    logger.info(f"  전체 test 샘플 수: {len(test_samples)}개")

    SEASON_MAP = {
        "primavera": "spring",
        "estate": "summer",
        "autunno": "autumn",
        "inverno": "winter",
    }

    PROMPT = (
        "You are an expert in Armocromia personal color analysis. "
        "Carefully examine the person's skin tone, hair color, and eye color in the image. "
        "Classify their seasonal palette into exactly one of: Spring, Summer, Autumn, Winter. "
        "Respond with ONLY the single season word. Do not explain."
    )

    correct = 0
    valid = 0
    failed_api = 0
    class_correct = {s: 0 for s in SEASON_MAP.values()}
    class_total   = {s: 0 for s in SEASON_MAP.values()}

    for _, row in tqdm(test_samples.iterrows(), total=len(test_samples), desc="Armocromia"):
        img_path = BASE_DIR / str(row["path_rgb_original"])
        if not img_path.exists():
            logger.debug(f"이미지 없음: {img_path}")
            continue

        gt = SEASON_MAP.get(str(row["class"]).strip().lower(), "")
        if not gt:
            continue

        pred, _ = ask_vlm(PROMPT, img_path, task_name="Armocromia")

        if pred is None:
            failed_api += 1
            continue

        valid += 1
        class_total[gt] += 1

        if is_correct(pred, gt, "Armocromia"):
            correct += 1
            class_correct[gt] += 1

        logger.debug(f"  GT: {gt} | Pred: {pred} | {'✅' if is_correct(pred, gt) else '❌'}")

    overall_acc = (correct / valid * 100) if valid > 0 else 0.0

    per_class = {}
    for season in SEASON_MAP.values():
        n = class_total[season]
        per_class[season] = (class_correct[season] / n * 100) if n > 0 else 0.0

    return {
        "accuracy": overall_acc,
        "correct": correct,
        "valid": valid,
        "failed_api": failed_api,
        "per_class": per_class,
    }


# ====================================================
# 6. 리포트 출력
# ====================================================
def print_report(cb_scores: dict, armo_result: dict):
    print("\n" + "=" * 60)
    print("📊  Qwen2.5-VL 프로젝트 통합 벤치마크 리포트")
    print("=" * 60)

    print("\n[ ColorBench ]")
    print(f"  {'Task':<22} {'Accuracy':>8}  {'Valid':>10}  {'API Fail':>8}")
    print("  " + "-" * 58)
    cb_avg = []
    for task, info in cb_scores.items():
        print(
            f"  {task:<22} {info['accuracy']:>7.2f}%"
            f"  {info['correct']:>5}/{info['valid']:<5}"
            f"  (전체 {info['total']}개, 실패: {info['failed_api']})"
        )
        cb_avg.append(info["accuracy"])
    if cb_avg:
        print(f"\n  {'ColorBench 평균':<22} {sum(cb_avg)/len(cb_avg):>7.2f}%")

    print("\n[ Deep-Armocromia ]")
    if armo_result:
        print(f"  전체 정확도  : {armo_result['accuracy']:.2f}%  ({armo_result['correct']}/{armo_result['valid']}, API 실패: {armo_result['failed_api']})")
        print("  클래스별 정확도:")
        for season, acc in armo_result.get("per_class", {}).items():
            print(f"    {season:<10}: {acc:.2f}%")
    else:
        print("  ⚠️ Armocromia 결과 없음")

    print("\n" + "=" * 60)


# ====================================================
# 7. 실행
# ====================================================
if __name__ == "__main__":
    cb_scores   = eval_colorbench_full()
    armo_result = eval_armocromia()
    print_report(cb_scores, armo_result)

2026-05-14 20:05:10,585 [INFO] 🚀 [1/2] ColorBench 종합 평가 시작 (전체 데이터셋)...
2026-05-14 20:05:10,881 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/umd-zhou-lab/ColorBench/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-05-14 20:05:10,898 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/umd-zhou-lab/ColorBench/18863f1ae0fbbd90553f3caa5d09bef5993c04b1/README.md "HTTP/1.1 200 OK"
2026-05-14 20:05:11,117 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/umd-zhou-lab/ColorBench/resolve/18863f1ae0fbbd90553f3caa5d09bef5993c04b1/ColorBench.py "HTTP/1.1 404 Not Found"
2026-05-14 20:05:11,799 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/umd-zhou-lab/ColorBench/umd-zhou-lab/ColorBench.py "HTTP/1.1 404 Not Found"
2026-05-14 20:05:12,021 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/umd-zhou-lab/ColorBench/resolve/18863f1ae0fbbd90553f3caa5d09bef5993c04b1/.huggingface.yaml "HTTP/1.1 40

KeyboardInterrupt: 